Transformers 라이브러리

인코더 기반 모델: 예시 모델 Multiple Choice
- 문제 예시) 여러개 입력이 주어졌을때 주어진 문장중 옳은 문장을 고르는 객관식 문제, 
- 입력 프롬프트 한개와 그에 대응하는 n개 후보로 구성됨
- 프롬프트 + 답변1, 프롬프트 + 답변2,...프롬프트 + 답변n 목록을 모두 입력 받아 후보중 무엇이 정답인지 출력

1. klue/bert-base 기본 모델: BertForMultipleChoice
- 추론 : 수능 국어 문제 풀이 - 5지선다형(보기 5개)

2. klue/bert-base 기본 모델: BertForMultipleChoice
- 파인튜닝 : 수능 국어 문제 풀이 - 5지선다형(보기 5개)
- 학습용 데이터셋 준비: 데이터셋(수능 문제 풀이 HAERAE-HUB/csatqa)
- 데이터셋 전처리(토크나이징), 샘플링
- 트레이너(Trainer) 설정 
- 학습 실행
- 모델 저장
- 모델 로드 및 모델 추론
- 허깅페이스 모델 업로드

In [2]:
# 인코더 기반 모델
# 1. klue/bert-base 기본 모델: BertForMultipleChoice
# 예를 들어, "지문"과 "보기 1, 2, 3, 4, 5"가 주어졌을 때 각각의 보기가 정답일 확률(점수)을 계산하여 
# 가장 높은 점수를 가진 보기를 선택할 수 있게 만들어 준다
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice

# 모델 이름 지정 및 사전학습 모델 로드
model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMultipleChoice.from_pretrained(model_name)

# 모델 출력 확인
print(f"model : {model}")

/Users/ai/deep-learning/LLM/rag_pdf_qdrant/venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of BertForMultipleChoice were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model : BertForMultipleChoice(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
# 인코더 기반 모델
# 1. klue/bert-base 기본 모델: BertForMultipleChoice
# 추론 : 수능 국어 문제 풀이 - 5지선다형(보기 5개)


In [13]:
# 인코더 기반 모델
# 1. klue/bert-base 기본 모델: BertForMultipleChoice
# 데이터셋(수능 문제 풀이 HAERAE-HUB/csatqa)
# - 수능 국어 문제 데이터셋으로, 5개 정답 중 하나를 고르는 문제
# - 예를 들어, "지문"과 "보기 1, 2, 3, 4, 5"가 주어졌을 때 각각의 보기가 정답일 확률(점수)을 계산하여 
# - 가장 높은 점수를 가진 보기를 선택할 수 있게 만들어 준다
from datasets import load_dataset

dataset = load_dataset("HAERAE-HUB/csatqa", "full")
print(f"dataset[test] : {dataset['test']}")
print(f"dataset[test][0] : {dataset['test'][0]}")

dataset[test] : Dataset({
    features: ['question', 'context', 'option#1', 'option#2', 'option#3', 'option#4', 'option#5', 'gold', 'category', 'human_performance'],
    num_rows: 936
})
dataset[test][0] : {'question': ' 이 이야기에서 얻을 수 있는 교훈으로 가장 적절한 것은?', 'context': '이제 한 편의 이야기를 들려 드립니다. 잘 듣고 물음에 답하십시오.\n자, 여러분! 안녕하십니까? 오늘은 제가 어제 꾼 꿈 이야기 하날 들려 드리겠습니다. 전 꿈속에서 낯선 거리를 걷고 있었습니다. 그러다가 홍미로운 간판을 발견했답니다. 행 복을 파는 가게. 그렇게 쓰여 있었습니다. 전 호기심으로 문을 열고 들어갔답니다. 그곳 에서는 한 노인이 물건을 팔고 있었습니다. 전 잠시 머뭇거리다가 노인에게 다가가서 물 었습니다. 여기서는 무슨 물건을 파느냐고요. 노인은 미소를 지으며, 원하는 것은 뭐든 다 살 수 있다고 말했습니다. 저는 제 귀를 의심했습니다. \'무엇이든 다?\' 전 무엇을 사야 할까 생각하다가 말했답니다. "사랑, 부귀 그리고 지혜하고 건강도 사고 싶습니다. 저 자신뿐 아니라 우리 가족 모두 를 위해서요. 지금 바로 살 수 있나요?" 그러자 노인은 빙긋이 웃으며 대답했습니다. "젊은이, 한번 잘 보게나. 여기에서 팔고 있는 것은 무르익은 과일이 아니라 씨앗이라 네. 앞으로 좋은 열매를 맺으려면 이 씨앗들을 잘 가꾸어야 할 걸세."', 'option#1': '새로운 세계에 대한 열망을 가져야 한다.', 'option#2': '주어진 기회를 능동적으로 활용해야 한다.', 'option#3': '큰 것을 얻으려면 작은 것은 버려야 한다.', 'option#4': '물질적 가치보다 정신적 가치를 중시해야 한다.', 'option#5': '소망하는 바를 성취하기 위해서는 노력을 해야 한다